In [1]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [4]:
os.makedirs(SSP_DIR_PATH, exist_ok=True)

In [5]:
edau = EDAUtils()

## Pull SSP run from AWS S3 and load other important files


In [6]:
aws_config = edau.read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
profile_name = aws_config["profile_name"]
bucket_name = aws_config["bucket_name"]
# Set your profile
session = boto3.Session(profile_name=profile_name)

# Create an S3 client or resource
s3 = session.resource('s3')

# Define folder prefix
prefix = 'sisepuede_run_2025-01-14T17;04;06.975301_output_database/'  # this is like the "folder" in S3

In [7]:
# Local destination
destination = os.path.join(SSP_DIR_PATH, prefix.strip('/'))
if os.path.exists(destination) and os.listdir(destination):
    print(f"Destination '{destination}' already exists and is not empty. Skipping download.")
else:
    os.makedirs(destination, exist_ok=True)
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=prefix):
        if obj.key.endswith('/'):  # skip directories
            continue
        target_path = os.path.join(destination, os.path.basename(obj.key))
        bucket.download_file(obj.key, target_path)
        print(f"Downloaded: {obj.key}")

Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/ANALYSIS_METADATA.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/ATTRIBUTE_DESIGN.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/ATTRIBUTE_PRIMARY.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/ATTRIBUTE_STRATEGY.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/MODEL_BASE_INPUT_DATABASE.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/MODEL_INPUT.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/MODEL_OUTPUT.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/WIDE_INPUTS_OUTPUTS.csv
Downloaded: sisepuede_run_2025-01-14T17;04;06.975301_output_database/lhs_sample_group_experiments.csv


In [9]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, prefix.strip('/'))

In [10]:
lhc_trials_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "lhs_sample_group_experiments.csv"))
lhc_trials_df

,1,2,3,4,5,6,7,8,9,10,...,1472,1473,1474,1475,1476,1477,1478,1480,1481,future_id
0,0.952436,0.360397,0.202990,0.995748,0.369859,0.919920,0.478450,0.445798,0.490988,0.510280,...,0.985425,0.482783,0.183188,0.364889,0.005297,0.223598,0.927810,0.036368,0.933822,1
1,0.304955,0.204875,0.540658,0.800192,0.206396,0.777352,0.805727,0.536048,0.779857,0.623335,...,0.437877,0.149747,0.308130,0.780839,0.784662,0.742266,0.147130,0.081654,0.491202,2
2,0.609262,0.366443,0.601808,0.916668,0.386475,0.181162,0.138910,0.768925,0.884425,0.232722,...,0.360238,0.348950,0.165501,0.813845,0.950005,0.342632,0.053104,0.316417,0.173655,3
3,0.815313,0.487444,0.323954,0.634299,0.064625,0.926815,0.633805,0.439346,0.027114,0.532020,...,0.060470,0.660562,0.621883,0.911838,0.030974,0.912003,0.150989,0.246583,0.353195,4
4,0.973878,0.842718,0.884382,0.483039,0.455502,0.003343,0.571994,0.108021,0.346714,0.892123,...,0.910718,0.621574,0.244454,0.927056,0.561529,0.611280,0.552761,0.125694,0.147930,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.218189,0.886941,0.569155,0.107766,0.955775,0.380014,0.569711,0.312233,0.791412,0.300450,...,0.669261,0.883331,0.431040,0.045121,0.790492,0.081530,0.351688,0.746613,0.853002,996
996,0.510025,0.235856,0.847465,0.138888,0.182086,0.453749,0.242018,0.215693,0.657012,0.808412,...,0.538035,0.037013,0.528031,0.079728,0.918024,0.653280,0.161674,0.033243,0.257661,997
997,0.276479,0.612911,0.642697,0.952037,0.176638,0.411246,0.169671,0.694365,0.606286,0.678398,...,0.262631,0.657597,0.243538,0.603350,0.377625,0.316224,0.615736,0.996441,0.704642,998
998,0.600573,0.553323,0.306380,0.464389,0.834916,0.786519,0.717580,0.168263,0.009808,0.855097,...,0.069343,0.293393,0.632635,0.168413,0.611178,0.295950,0.813797,0.934377,0.473035,999


In [11]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "WIDE_INPUTS_OUTPUTS.csv"))
wide_inputs_outputs_df

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_fruits_tonne_ha,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha
0,275275,louisiana,7,0,352056.299497,65286.215373,76.761572,75770.572219,6423.938390,1.104616e+06,...,18.989020,11.831723,2.782454,6.177415,0,3.474771,8.253027,87.719298,41.144328,31.476288
1,275275,louisiana,8,0,349848.854439,64876.861148,76.280265,75295.479526,6383.659346,1.097689e+06,...,18.791974,11.736001,2.699011,5.189029,0,2.688411,8.475414,83.271559,41.748977,31.761360
2,275275,louisiana,9,0,347657.028850,64470.403437,75.802365,74823.748501,6343.665309,1.090812e+06,...,18.594927,11.640280,2.615567,6.319971,0,3.416092,8.422668,89.334930,42.353627,32.046432
3,275275,louisiana,10,0,345480.707102,64066.820799,75.327844,74355.354257,6303.954170,1.083984e+06,...,18.397880,11.544559,2.532124,6.299606,0,3.415907,8.422736,90.402061,42.958276,32.331504
4,275275,louisiana,11,0,343319.774453,63666.091953,74.856679,73890.272100,6264.523834,1.077204e+06,...,18.200833,11.448838,2.448681,6.279241,0,3.415722,8.422804,91.469193,43.562926,32.616576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28560,275293,louisiana,31,0,294221.428184,63220.177553,89.937159,81027.978802,16195.145410,1.078609e+06,...,15.352838,10.364332,0.639586,6.489255,0,3.802796,9.390390,128.685483,63.702379,43.496713
28561,275293,louisiana,32,0,292599.377554,63909.784176,92.081486,82305.169476,17949.679259,1.087869e+06,...,15.332242,10.370063,0.599125,6.513671,0,3.823289,9.441271,129.944000,64.365588,43.882292
28562,275293,louisiana,33,0,291148.417580,64642.940250,94.277370,83613.385961,19878.507594,1.097669e+06,...,15.323540,10.381572,0.563700,6.539317,0,3.843794,9.492147,131.138110,64.992304,44.250665
28563,275293,louisiana,34,0,289866.317272,65408.965880,96.504382,84938.693208,21970.474633,1.107899e+06,...,15.326730,10.398858,0.533311,6.566192,0,3.864310,9.543020,132.267813,65.582526,44.601833


## Data Cleaning

In [12]:
# Filter to only emission columns avoiding "subsector" total columns
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + [c for c in wide_inputs_outputs_df.columns if "emission_" in c and "subsector_total" not in c]]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_c2f6_ippu_product_use_product_use_ods_other,emission_co2e_c2f6_ippu_production_chemicals,emission_co2e_c2f6_ippu_production_electronics,emission_co2e_c2f6_ippu_production_metals,emission_co2e_c2h3f3_ippu_product_use_product_use_ods_refrigeration,emission_co2e_c2h3f3_ippu_production_chemicals,emission_co2e_c2hf5_ippu_product_use_product_use_ods_other,emission_co2e_c2hf5_ippu_product_use_product_use_ods_refrigeration,...,emission_co2e_pfcs_ippu_production_chemicals,emission_co2e_pfcs_ippu_production_electronics,emission_co2e_pfcs_ippu_production_other_product_manufacturing,emission_co2e_sf6_ippu_production_chemicals,emission_co2e_sf6_ippu_production_electronics,emission_co2e_sf6_ippu_production_metals,emission_co2e_sf6_ippu_production_other_product_manufacturing,emission_nongas_fgtv_kt_nmvoc_fuel_coal,emission_nongas_fgtv_kt_nmvoc_fuel_natural_gas,emission_nongas_fgtv_kt_nmvoc_fuel_oil
0,275275,7,0.134551,0,0.003159,0,0.137710,0,0.068855,0.068855,...,0,0,0,0,0,0,0,0,16.298474,26.879367
1,275275,8,0.136654,0,0.003139,0,0.139862,0,0.069931,0.069931,...,0,0,0,0,0,0,0,0,16.749565,24.067988
2,275275,9,0.138907,0,0.003121,0,0.142168,0,0.071084,0.071084,...,0,0,0,0,0,0,0,0,14.880639,26.145386
3,275275,10,0.141280,0,0.003105,0,0.144596,0,0.072298,0.072298,...,0,0,0,0,0,0,0,0,14.929146,26.033956
4,275275,11,0.143748,0,0.003092,0,0.147123,0,0.073561,0.073561,...,0,0,0,0,0,0,0,0,14.990676,25.922457


In [13]:
[c for c in la_emissions_df.columns if "subsector_total" in c]

[]

## Transform time series format into single-row format

In [14]:
# get the columns that have negative values
cols_with_neg_values = la_emissions_df.columns[la_emissions_df.min() < 0]
cols_with_neg_values

Index(['emission_co2e_ch4_trww_treated_septic_treatment',
       'emission_co2e_co2_ccsq_direct_air_capture',
       'emission_co2e_co2_entc_generation_pp_gas_ccs',
       'emission_co2e_co2_frst_harvested_wood_products',
       'emission_co2e_co2_frst_sequestration_mangroves',
       'emission_co2e_co2_frst_sequestration_primary',
       'emission_co2e_co2_frst_sequestration_secondary',
       'emission_co2e_co2_soil_soc_mineral_soils'],
      dtype='object')

In [15]:
la_emissions_df[cols_with_neg_values].describe()

,emission_co2e_ch4_trww_treated_septic_treatment,emission_co2e_co2_ccsq_direct_air_capture,emission_co2e_co2_entc_generation_pp_gas_ccs,emission_co2e_co2_frst_harvested_wood_products,emission_co2e_co2_frst_sequestration_mangroves,emission_co2e_co2_frst_sequestration_primary,emission_co2e_co2_frst_sequestration_secondary,emission_co2e_co2_soil_soc_mineral_soils
count,28565.000000,28565.000000,2.856500e+04,28565.000000,28565.000000,28565.000000,28565.000000,28565.000000
mean,0.002745,-7.091305,2.265410e-05,-6.770839,-28.109623,-8.410377,-2.980809,0.076564
std,0.001592,7.136138,5.790369e-04,4.355612,1.396470,0.020771,0.295708,0.107779
min,-0.002781,-31.136960,-9.030113e-26,-21.146151,-32.292858,-8.453841,-6.917437,-0.231446
25%,0.001702,-11.298574,0.000000e+00,-9.067847,-29.067382,-8.424021,-3.133331,-0.000752
50%,0.003195,-4.955811,0.000000e+00,-6.491411,-27.907248,-8.415556,-2.979060,0.085273
75%,0.004166,-0.952618,0.000000e+00,-4.029517,-26.865639,-8.403412,-2.793632,0.183260
max,0.004706,0.000000,2.601375e-02,3.575664,-26.301658,-8.264373,-1.662824,0.211129


In [16]:
negative_cols = [c for c in cols_with_neg_values if "ccsq" in c or "frst" in c]
negative_cols

['emission_co2e_co2_ccsq_direct_air_capture',
 'emission_co2e_co2_frst_harvested_wood_products',
 'emission_co2e_co2_frst_sequestration_mangroves',
 'emission_co2e_co2_frst_sequestration_primary',
 'emission_co2e_co2_frst_sequestration_secondary']

In [17]:
# Get the columns that have positive values
positive_cols = [col for col in la_emissions_df.columns if col not in negative_cols and col not in ["primary_id", "time_period"]]
positive_cols

['emission_co2e_c2f6_ippu_product_use_product_use_ods_other',
 'emission_co2e_c2f6_ippu_production_chemicals',
 'emission_co2e_c2f6_ippu_production_electronics',
 'emission_co2e_c2f6_ippu_production_metals',
 'emission_co2e_c2h3f3_ippu_product_use_product_use_ods_refrigeration',
 'emission_co2e_c2h3f3_ippu_production_chemicals',
 'emission_co2e_c2hf5_ippu_product_use_product_use_ods_other',
 'emission_co2e_c2hf5_ippu_product_use_product_use_ods_refrigeration',
 'emission_co2e_c2hf5_ippu_production_chemicals',
 'emission_co2e_c3f8_ippu_production_chemicals',
 'emission_co2e_c3f8_ippu_production_electronics',
 'emission_co2e_c3h2f6_ippu_product_use_product_use_ods_other',
 'emission_co2e_c3h2f6_ippu_product_use_product_use_ods_refrigeration',
 'emission_co2e_c3h3f5_ippu_product_use_product_use_ods_other',
 'emission_co2e_c3h3f5_ippu_product_use_product_use_ods_refrigeration',
 'emission_co2e_c3hf7_ippu_product_use_product_use_ods_other',
 'emission_co2e_c3hf7_ippu_product_use_product_use

In [18]:
la_emissions_totals_df = la_emissions_df.copy()
la_emissions_totals_df["total_positive_emissions"] = la_emissions_totals_df[positive_cols].sum(axis=1)
la_emissions_totals_df["total_negative_emissions"] = la_emissions_totals_df[negative_cols].sum(axis=1)

In [19]:
la_emissions_totals_df = la_emissions_totals_df[["primary_id", "time_period", "total_positive_emissions", "total_negative_emissions"]]
la_emissions_totals_df.head()

,primary_id,time_period,total_positive_emissions,total_negative_emissions
0,275275,7,298.426252,-36.545260
1,275275,8,289.135257,-38.818142
2,275275,9,305.747084,-40.622659
3,275275,10,305.609580,-42.118691
4,275275,11,305.528194,-43.407064


In [20]:
# Filter out rows with time_period <= 30
la_emissions_totals_df = la_emissions_totals_df[la_emissions_totals_df["time_period"] > 30]
la_emissions_totals_df.head(7)

,primary_id,time_period,total_positive_emissions,total_negative_emissions
24,275275,31,48.087540,-68.177510
25,275275,32,44.607433,-69.843305
26,275275,33,41.300103,-71.559940
27,275275,34,37.780607,-73.318381
28,275275,35,34.652544,-75.105459
53,275379,31,112.492853,-49.041584
54,275379,32,111.172956,-49.287741


In [21]:
# aggregate data by primary_id and region by summing the emissions
la_emissions_df_agg = la_emissions_totals_df.groupby(["primary_id"]).sum().reset_index()
la_emissions_df_agg

,primary_id,time_period,total_positive_emissions,total_negative_emissions
0,275275,165,206.428228,-358.004594
1,275276,165,898.896088,-329.706438
2,275277,165,641.742705,-232.744433
3,275278,165,394.675852,-334.428479
4,275279,165,755.876824,-314.675683
...,...,...,...,...
980,276271,165,645.016903,-367.411380
981,276272,165,919.296399,-421.618796
982,276273,165,825.581632,-317.761556
983,276274,165,811.346650,-292.933590


In [22]:
# Drop time period column as it is no longer needed
la_emissions_df_agg = la_emissions_df_agg.drop(columns=["time_period"], errors='ignore')
la_emissions_df_agg.describe()

,primary_id,total_positive_emissions,total_negative_emissions
count,985.000000,9.850000e+02,985.000000
mean,275776.930964,1.047967e+06,-327.558584
std,289.326466,1.451735e+07,48.354918
min,275275.000000,2.064282e+02,-445.411052
25%,275528.000000,6.230532e+02,-362.884017
50%,275778.000000,7.061427e+02,-327.041288
75%,276027.000000,8.164495e+02,-292.407466
max,276275.000000,3.037559e+08,-214.071578


## Merge emissions data with lhs samples

In [23]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
0,275275,3,6002,0
1,275276,3,6002,1
2,275277,3,6002,2
3,275278,3,6002,3
4,275279,3,6002,4
...,...,...,...,...
996,276271,3,6002,996
997,276272,3,6002,997
998,276273,3,6002,998
999,276274,3,6002,999


In [24]:
la_emissions_df_w_future_id = la_emissions_df_agg.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_emissions_df_w_future_id = la_emissions_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_emissions_df_w_future_id

,primary_id,total_positive_emissions,total_negative_emissions,future_id
0,275275,206.428228,-358.004594,0
1,275276,898.896088,-329.706438,1
2,275277,641.742705,-232.744433,2
3,275278,394.675852,-334.428479,3
4,275279,755.876824,-314.675683,4
...,...,...,...,...
980,276271,645.016903,-367.411380,996
981,276272,919.296399,-421.618796,997
982,276273,825.581632,-317.761556,998
983,276274,811.346650,-292.933590,999


In [25]:
merged_df = pd.merge(lhc_trials_df, la_emissions_df_w_future_id, on="future_id", how="inner")

# rearrange columns to have future_id and primary_id at the front
cols_order = ["future_id", "primary_id"] + [col for col in merged_df.columns if col not in ["future_id", "primary_id"]]
merged_df = merged_df[cols_order]
merged_df

,future_id,primary_id,1,2,3,4,5,6,7,8,...,1473,1474,1475,1476,1477,1478,1480,1481,total_positive_emissions,total_negative_emissions
0,1,275276,0.952436,0.360397,0.202990,0.995748,0.369859,0.919920,0.478450,0.445798,...,0.482783,0.183188,0.364889,0.005297,0.223598,0.927810,0.036368,0.933822,898.896088,-329.706438
1,2,275277,0.304955,0.204875,0.540658,0.800192,0.206396,0.777352,0.805727,0.536048,...,0.149747,0.308130,0.780839,0.784662,0.742266,0.147130,0.081654,0.491202,641.742705,-232.744433
2,3,275278,0.609262,0.366443,0.601808,0.916668,0.386475,0.181162,0.138910,0.768925,...,0.348950,0.165501,0.813845,0.950005,0.342632,0.053104,0.316417,0.173655,394.675852,-334.428479
3,4,275279,0.815313,0.487444,0.323954,0.634299,0.064625,0.926815,0.633805,0.439346,...,0.660562,0.621883,0.911838,0.030974,0.912003,0.150989,0.246583,0.353195,755.876824,-314.675683
4,5,275280,0.973878,0.842718,0.884382,0.483039,0.455502,0.003343,0.571994,0.108021,...,0.621574,0.244454,0.927056,0.561529,0.611280,0.552761,0.125694,0.147930,628.746867,-360.795994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
979,996,276271,0.218189,0.886941,0.569155,0.107766,0.955775,0.380014,0.569711,0.312233,...,0.883331,0.431040,0.045121,0.790492,0.081530,0.351688,0.746613,0.853002,645.016903,-367.411380
980,997,276272,0.510025,0.235856,0.847465,0.138888,0.182086,0.453749,0.242018,0.215693,...,0.037013,0.528031,0.079728,0.918024,0.653280,0.161674,0.033243,0.257661,919.296399,-421.618796
981,998,276273,0.276479,0.612911,0.642697,0.952037,0.176638,0.411246,0.169671,0.694365,...,0.657597,0.243538,0.603350,0.377625,0.316224,0.615736,0.996441,0.704642,825.581632,-317.761556
982,999,276274,0.600573,0.553323,0.306380,0.464389,0.834916,0.786519,0.717580,0.168263,...,0.293393,0.632635,0.168413,0.611178,0.295950,0.813797,0.934377,0.473035,811.346650,-292.933590


In [26]:
var_traj_df = pd.read_csv(os.path.join(MAPPING_DIR_PATH, "var_trajgroups_experiment.csv"))
var_traj_df.tail(10)

,variable_trajectory_group
37,38
38,39
39,40
40,41
41,42
42,43
43,44
44,45
45,46
46,47


In [27]:
# Convert the variable_trajectory_group column to list of strings
relevant_lhs_cols = var_traj_df["variable_trajectory_group"].values
relevant_lhs_cols = [str(col) for col in relevant_lhs_cols.tolist()]
# relevant_lhs_cols

In [28]:
df_cols = merged_df.columns.tolist()

# Filter the relevant_lhs_cols to only include those that are in df_cols
relevant_lhs_cols = [col for col in relevant_lhs_cols if col in df_cols]

In [29]:
# filter merged_df to keep only relevant columns
cols_to_keep = ["future_id", "primary_id"] + list(relevant_lhs_cols) + ["total_positive_emissions", "total_negative_emissions"]
merged_df_filtered = merged_df[cols_to_keep]

In [30]:
merged_df_filtered

,future_id,primary_id,1,2,3,4,5,6,7,8,...,40,41,42,43,44,45,46,47,total_positive_emissions,total_negative_emissions
0,1,275276,0.952436,0.360397,0.202990,0.995748,0.369859,0.919920,0.478450,0.445798,...,0.289038,0.996557,0.053597,0.175103,0.379422,0.438093,0.149362,0.869757,898.896088,-329.706438
1,2,275277,0.304955,0.204875,0.540658,0.800192,0.206396,0.777352,0.805727,0.536048,...,0.536533,0.385371,0.246165,0.101025,0.717013,0.368791,0.774941,0.483352,641.742705,-232.744433
2,3,275278,0.609262,0.366443,0.601808,0.916668,0.386475,0.181162,0.138910,0.768925,...,0.533412,0.601590,0.122444,0.653845,0.216259,0.714853,0.881270,0.581230,394.675852,-334.428479
3,4,275279,0.815313,0.487444,0.323954,0.634299,0.064625,0.926815,0.633805,0.439346,...,0.264120,0.938985,0.592915,0.564895,0.027725,0.138890,0.820709,0.250616,755.876824,-314.675683
4,5,275280,0.973878,0.842718,0.884382,0.483039,0.455502,0.003343,0.571994,0.108021,...,0.430582,0.492564,0.432240,0.662820,0.179231,0.631375,0.395960,0.661637,628.746867,-360.795994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
979,996,276271,0.218189,0.886941,0.569155,0.107766,0.955775,0.380014,0.569711,0.312233,...,0.004788,0.319760,0.567859,0.873065,0.309462,0.213959,0.531121,0.413153,645.016903,-367.411380
980,997,276272,0.510025,0.235856,0.847465,0.138888,0.182086,0.453749,0.242018,0.215693,...,0.695228,0.368169,0.356410,0.584330,0.195851,0.217361,0.920388,0.402516,919.296399,-421.618796
981,998,276273,0.276479,0.612911,0.642697,0.952037,0.176638,0.411246,0.169671,0.694365,...,0.326525,0.828078,0.653782,0.429175,0.977175,0.914779,0.307569,0.506892,825.581632,-317.761556
982,999,276274,0.600573,0.553323,0.306380,0.464389,0.834916,0.786519,0.717580,0.168263,...,0.398381,0.757483,0.826523,0.961181,0.882943,0.900195,0.448056,0.664405,811.346650,-292.933590


## Add variable names to lhs columns

In [31]:
var_specifications_df = pd.read_csv(os.path.join(MAPPING_DIR_PATH, "variable_specification_to_sample_group.csv"))
# Filter var_specifications_df by sample_group in relevant_lhs_cols
relevant_lhs_cols = [int(col) for col in relevant_lhs_cols]
var_specifications_df = var_specifications_df[var_specifications_df["sample_group"].isin(relevant_lhs_cols)]
var_specifications_df = var_specifications_df.sort_values(by="sample_group", ascending=True)

In [32]:
def process_variable_prefix(df):
    result = []
    for group, group_df in df.groupby('sample_group'):
        variables = group_df['variable_specification'].tolist()
        if len(variables) == 1:
            prefix = variables[0]
        else:
            prefix = os.path.commonprefix(variables)
            # Clean trailing underscores
            prefix = prefix.rstrip('_')
            
        prefix = f"group_{group}_{prefix}"
        result.append({'sample_group': group, 'variable_prefix': prefix})
    return pd.DataFrame(result)

prefix_df = process_variable_prefix(var_specifications_df)
prefix_df

,sample_group,variable_prefix
0,1,group_1_ef_lvst_entferm
1,2,group_2_yf_agrc
2,3,group_3_demscalar_soil
3,4,group_4_demscalar_ippu
4,5,group_5_efficfactor_enfu_industrial_energy_fuel
5,6,group_6_elecfuelefficiency_trns
6,7,group_7_factor_waso_waste_per_capita_scalar_food
7,8,group_8_frac_fgtv
8,9,group_9_frac_agrc_agriculture_production_lost
9,10,group_10_frac_agrc


In [33]:
var_specifications_df[var_specifications_df["sample_group"].isin([13, 40])]

,variable_specification,sample_group
560,elasticity_agrc_sugar_cane_demand_to_income,13
876,frac_gnrl_eating_red_meat,13
1177,frac_lndu_proportion_grasslands_pasture,40
2020,pij_lndu_grasslands_to_forests_primary,40
2021,pij_lndu_grasslands_to_forests_secondary,40
2022,pij_lndu_grasslands_to_grasslands,40
2024,pij_lndu_grasslands_to_settlements,40
2025,pij_lndu_grasslands_to_wetlands,40
2023,pij_lndu_grasslands_to_other,40
2019,pij_lndu_grasslands_to_forests_mangroves,40


In [34]:
prefix_df.loc[prefix_df["sample_group"] == 13, "variable_prefix"] = "group_13_frac_gnrl_eating_red_meats+"
prefix_df.loc[prefix_df["sample_group"] == 40, "variable_prefix"] = "group_40_pij_lndu_grasslands+"

prefix_df = prefix_df.sort_values(by="variable_prefix", ascending=True)
prefix_df

,sample_group,variable_prefix
9,10,group_10_frac_agrc
10,12,group_12_frac_enfu_transmission_loss_fuel_elec...
11,13,group_13_frac_gnrl_eating_red_meats+
12,14,group_14_frac_ippu_cement_clinker
13,15,group_15_frac_ippu_production_with_co2_capture
14,16,group_16_frac_lvst_mm
15,17,group_17_frac_trns_fuelmix
16,18,group_18_frac_trns_mtkm_dem_freight
17,19,group_19_frac_trns_pkm_dem
0,1,group_1_ef_lvst_entferm


In [35]:
# Let's use the prefix_df to rename the columns in merged_df_filtered
def rename_columns_with_prefix(merged_df, prefix_df):
    df = merged_df.copy()
    for _, row in prefix_df.iterrows():
        sample_group = row['sample_group']
        variable_prefix = row['variable_prefix']
        cols_to_rename = [col for col in df.columns if col.startswith(str(sample_group))]
        for col in cols_to_rename:
            new_col_name = f"{variable_prefix}"
            df = df.rename(columns={col: new_col_name})
    return df
merged_df_filtered_w_prefix = rename_columns_with_prefix(merged_df_filtered, prefix_df)

In [36]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_lvst_entferm,group_2_yf_agrc,group_3_demscalar_soil,group_4_demscalar_ippu,group_5_efficfactor_enfu_industrial_energy_fuel,group_6_elecfuelefficiency_trns,group_7_factor_waso_waste_per_capita_scalar_food,group_8_frac_fgtv,...,group_40_pij_lndu_grasslands+,group_41_pij_lndu_other_to,group_42_pij_lndu_settlements_to,group_43_pij_lndu_wetlands_to,group_44_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_45_scalar_lvst_carrying_capacity,group_46_scalar_scoe_appliance_energy_demand,group_47_scalar_scoe_heat_energy_demand,total_positive_emissions,total_negative_emissions
0,1,275276,0.952436,0.360397,0.202990,0.995748,0.369859,0.919920,0.478450,0.445798,...,0.289038,0.996557,0.053597,0.175103,0.379422,0.438093,0.149362,0.869757,898.896088,-329.706438
1,2,275277,0.304955,0.204875,0.540658,0.800192,0.206396,0.777352,0.805727,0.536048,...,0.536533,0.385371,0.246165,0.101025,0.717013,0.368791,0.774941,0.483352,641.742705,-232.744433
2,3,275278,0.609262,0.366443,0.601808,0.916668,0.386475,0.181162,0.138910,0.768925,...,0.533412,0.601590,0.122444,0.653845,0.216259,0.714853,0.881270,0.581230,394.675852,-334.428479
3,4,275279,0.815313,0.487444,0.323954,0.634299,0.064625,0.926815,0.633805,0.439346,...,0.264120,0.938985,0.592915,0.564895,0.027725,0.138890,0.820709,0.250616,755.876824,-314.675683
4,5,275280,0.973878,0.842718,0.884382,0.483039,0.455502,0.003343,0.571994,0.108021,...,0.430582,0.492564,0.432240,0.662820,0.179231,0.631375,0.395960,0.661637,628.746867,-360.795994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
979,996,276271,0.218189,0.886941,0.569155,0.107766,0.955775,0.380014,0.569711,0.312233,...,0.004788,0.319760,0.567859,0.873065,0.309462,0.213959,0.531121,0.413153,645.016903,-367.411380
980,997,276272,0.510025,0.235856,0.847465,0.138888,0.182086,0.453749,0.242018,0.215693,...,0.695228,0.368169,0.356410,0.584330,0.195851,0.217361,0.920388,0.402516,919.296399,-421.618796
981,998,276273,0.276479,0.612911,0.642697,0.952037,0.176638,0.411246,0.169671,0.694365,...,0.326525,0.828078,0.653782,0.429175,0.977175,0.914779,0.307569,0.506892,825.581632,-317.761556
982,999,276274,0.600573,0.553323,0.306380,0.464389,0.834916,0.786519,0.717580,0.168263,...,0.398381,0.757483,0.826523,0.961181,0.882943,0.900195,0.448056,0.664405,811.346650,-292.933590


In [37]:
#save the merged DataFrame to a CSV file
merged_df_filtered_w_prefix.to_csv(os.path.join(TRAINING_DIR_PATH, "training_data_v3.csv"), index=False)